In [17]:
import pandas as pd

df = pd.read_csv('../dataset_filtered/articles.csv')

df

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,image_exists,image_path
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,True,dataset/images/010/0108775015.jpg
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,True,dataset/images/010/0108775044.jpg
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,True,dataset/images/010/0108775051.jpg
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",True,dataset/images/011/0110065001.jpg
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",True,dataset/images/011/0110065002.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105095,953450001,953450,5pk regular Placement1,302,Socks,Socks & Tights,1010014,Placement print,9,Black,...,Menswear,3,Menswear,26,Men Underwear,1021,Socks and Tights,Socks in a fine-knit cotton blend with a small...,True,dataset/images/095/0953450001.jpg
105096,953763001,953763,SPORT Malaga tank,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Ladieswear,1,Ladieswear,2,H&M+,1005,Jersey Fancy,Loose-fitting sports vest top in ribbed fast-d...,True,dataset/images/095/0953763001.jpg
105097,956217002,956217,Cartwheel dress,265,Dress,Garment Full body,1010016,Solid,9,Black,...,Ladieswear,1,Ladieswear,18,Womens Trend,1005,Jersey Fancy,"Short, A-line dress in jersey with a round nec...",True,dataset/images/095/0956217002.jpg
105098,957375001,957375,CLAIRE HAIR CLAW,72,Hair clip,Accessories,1010016,Solid,9,Black,...,Divided,2,Divided,52,Divided Accessories,1019,Accessories,Large plastic hair claw.,True,dataset/images/095/0957375001.jpg


In [18]:


sampled_df = (
    df.sort_values(['product_group_name'])
      .groupby('product_group_name', group_keys=False)
      .head(10)
)

sampled_df_test = (
    df.sort_values(['product_group_name'])
      .groupby('product_group_name', group_keys=False)
      .nth([10, 11, 12, 13])
)

features = sampled_df["product_group_name"].unique().tolist()

features

['Accessories',
 'Bags',
 'Cosmetic',
 'Fun',
 'Furniture',
 'Garment Full body',
 'Garment Lower body',
 'Garment Upper body',
 'Garment and Shoe care',
 'Interior textile',
 'Items',
 'Nightwear',
 'Shoes',
 'Socks & Tights',
 'Stationery',
 'Swimwear',
 'Underwear',
 'Underwear/nightwear',
 'Unknown']

### Create a WebDataset shard (images + metadata)

This will package the sampled rows (images + metadata) into a single `.tar` shard compatible with `webdataset` (keys: `<sampleid>.jpg`, `<sampleid>.json`).

Steps:
1. Ensure `sampled_df` exists (recompute if missing)
2. Resolve image paths
3. Write a shard tar file using `webdataset.TarWriter`
4. (Optional) Validate by reading it back
5. (Optional) Upload the `.tar` to Hugging Face Hub using `huggingface_hub` (`hf_hub_upload`)

You can later concatenate shards with naming like `shard-{000000..000123}.tar` for larger datasets.

In [19]:
# Build WebDataset shard
import json, io, hashlib
from pathlib import Path
from PIL import Image as PILImage
import webdataset as wds

assert 'sampled_df' in globals(), 'Run the sampling cell first.'
assert 'image_path' in sampled_df.columns, 'image_path column missing in sampled_df'

root = Path('..').resolve()  # project root
shards_dir = root / 'wds_shards/small_subsets'
shards_dir.mkdir(exist_ok=True, parents=True)
shard_path_train = shards_dir / 'sampled_train_000000.tar'
shard_path_test = shards_dir / 'sampled_test_000000.tar'
# Select metadata columns to include (exclude large free text if 

with wds.TarWriter(str(shard_path_train), compress=False) as sink: # type: ignore
    for i, row in sampled_df.iterrows():
        rel_img = row['image_path']
        img_file = root / rel_img
        if not img_file.exists():
            print('WARN missing image', img_file)
            continue
        # Resize to 256x256 and re-encode as JPEG bytes
        with PILImage.open(img_file) as im:
            im = im.convert('RGB')
            im = im.resize((256, 256), PILImage.LANCZOS)
            buf = io.BytesIO()
            im.save(buf, format='JPEG', quality=90, optimize=True)
            img_bytes = buf.getvalue()
        
        key = str(int(row['article_id']))
        
        
        sample = {
             "__key__": key,
            'jpg': img_bytes,
            'cls': features.index(row['product_group_name']),
        }
        sink.write(sample)

print('Shard written to', shard_path_train)
shard_path_train

Shard written to /Users/dan.popa/work/kaggle_getting_started/hm_article_sorter/wds_shards/small_subsets/sampled_train_000000.tar


PosixPath('/Users/dan.popa/work/kaggle_getting_started/hm_article_sorter/wds_shards/small_subsets/sampled_train_000000.tar')

In [20]:
# Build WebDataset shard
import json, io, hashlib
from pathlib import Path
from PIL import Image as PILImage
import webdataset as wds

assert 'sampled_df' in globals(), 'Run the sampling cell first.'
assert 'image_path' in sampled_df.columns, 'image_path column missing in sampled_df'

with wds.TarWriter(str(shard_path_test), compress=False) as sink: # type: ignore
    for i, row in sampled_df_test.iterrows():
        rel_img = row['image_path']
        img_file = root / rel_img
        if not img_file.exists():
            print('WARN missing image', img_file)
            continue
        
        # Resize to 256x256 and re-encode as JPEG bytes
        with PILImage.open(img_file) as im:
            im = im.convert('RGB')
            im = im.resize((256, 256), PILImage.LANCZOS)
            buf = io.BytesIO()
            im.save(buf, format='JPEG', quality=90, optimize=True)
            img_bytes = buf.getvalue()
        key = str(int(row['article_id']))
        
        # Write samples: key.jpg + key.json
        sample = {
             "__key__": key,
            'jpg': img_bytes,
            'cls': features.index(row['product_group_name']),
        }
        sink.write(sample)

print('Shard written to', shard_path_test)
shard_path_test

Shard written to /Users/dan.popa/work/kaggle_getting_started/hm_article_sorter/wds_shards/small_subsets/sampled_test_000000.tar


PosixPath('/Users/dan.popa/work/kaggle_getting_started/hm_article_sorter/wds_shards/small_subsets/sampled_test_000000.tar')

#### Validate and (optionally) upload the shard

The next cells show how to read a couple of samples back with `webdataset` and then upload the `.tar` file itself to the Hugging Face Hub repository (as an asset).

In [21]:
# Validate reading a couple of samples
import webdataset as wds

print('Reading back first 2 samples:')
for sample in wds.WebDataset(str(shard_path_train)).decode().to_tuple('jpg', 'cls'): # type: ignore
    # sample[0] is image bytes; sample[1] is json string
    print('Image bytes:', len(sample[0]), sample[1])
    

Reading back first 2 samples:
Image bytes: 8556 0
Image bytes: 6201 0
Image bytes: 5483 0
Image bytes: 6667 0
Image bytes: 5905 0
Image bytes: 5852 0
Image bytes: 4909 0
Image bytes: 6378 0
Image bytes: 9035 0
Image bytes: 7286 0
Image bytes: 7106 1
Image bytes: 8905 1
Image bytes: 9330 1
Image bytes: 8180 1
Image bytes: 7688 1
Image bytes: 6467 1
Image bytes: 7904 1
Image bytes: 8238 1
Image bytes: 7455 1
Image bytes: 7229 1
Image bytes: 11547 2
Image bytes: 10888 2
Image bytes: 15243 2
Image bytes: 7702 2
Image bytes: 4844 2
Image bytes: 8046 2
Image bytes: 7012 2
Image bytes: 12432 2
Image bytes: 6762 2
Image bytes: 18939 2
Image bytes: 11896 3
Image bytes: 8075 3
Image bytes: 6198 4
Image bytes: 6423 4
Image bytes: 7658 4
Image bytes: 5832 4
Image bytes: 8224 4
Image bytes: 6172 4
Image bytes: 5130 4
Image bytes: 8285 4
Image bytes: 5926 4
Image bytes: 6098 4
Image bytes: 19556 5
Image bytes: 7320 5
Image bytes: 14838 5
Image bytes: 7601 5
Image bytes: 16210 5
Image bytes: 17237 5


/Users/dan.popa/work/kaggle_getting_started/hm_article_sorter/.venv/lib/python3.11/site-packages/webdataset/compat.py:379: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn("WebDataset(shardshuffle=...) is None; set explicitly to False or a number")
